In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import math

In [2]:
def calc_stable_rank(tensor_W):
    """
    Computes Stable Rank: ||W||_F^2 / ||W||_2^2
    """
    if tensor_W.numel() == 0: return 1.0
    
    try:
        S = torch.linalg.svdvals(tensor_W.to(torch.float32))
        max_sigma = S[0]
        if max_sigma == 0: return 1.0
        
        frob_sq = torch.sum(S ** 2)
        spec_sq = max_sigma ** 2
        return (frob_sq / spec_sq).item()
    except Exception as e:
        print(f"SVD Error: {e}")
        return 1.0


class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank, alpha, dropout=0.0):
        super().__init__()
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Freeze original
        for param in self.original_layer.parameters():
            param.requires_grad = False
            
        in_dim = original_layer.in_features
        out_dim = original_layer.out_features
        
        # Init
        self.lora_A = nn.Parameter(torch.empty(rank, in_dim))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, rank))
        self.dropout = nn.Dropout(dropout)
        
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        
    def forward(self, x):
        base_out = self.original_layer(x)
        lora_out = (self.dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

    def get_update_matrix(self):
        return (self.lora_B @ self.lora_A) * self.scaling

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # 3 Layer MLP with ReLUs
        self.layers = nn.ModuleList([
            nn.Linear(input_dim, hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), # This will be the target
            nn.Linear(hidden_dim, output_dim)
        ])
        self.act = nn.ReLU()

    def forward(self, x):
        h = self.act(self.layers[0](x))
        h = self.act(self.layers[1](h))
        out = self.layers[2](h)
        return out

class LoRAMLPWrapper(nn.Module):
    def __init__(self, pretrained_mlp, rank, alpha=None):
        super().__init__()
        self.model = pretrained_mlp
        
        # Target the middle layer (index 1)
        target_layer = self.model.layers[1]
        
        # Freeze base
        for p in self.model.parameters(): p.requires_grad = False
            
        # Inject LoRA logic specifically for this layer
        # (We could replace the layer, but wrapping allows keeping the original structure cleaner for this toy exp)
        self.lora = LoRALayer(target_layer, rank=rank, alpha=alpha if alpha else rank*2)
        
        # Replace the layer in the module list so forward pass works automatically
        self.model.layers[1] = self.lora

    def forward(self, x):
        return self.model(x)
        
    def get_sr(self):
        return calc_stable_rank(self.lora.get_update_matrix())

In [3]:
def run_experiment(
    input_dim=64,
    hidden_dim=256,
    lora_rank=64,
    num_samples=5000,
    batch_size=128,
    pretrain_epochs=50,
    finetune_epochs=50,
    lr_pre=1e-3,
    lr_ft=2e-3,
    device="cuda" if torch.cuda.is_available() else "cpu"
):
    print(f"--- Starting Toy Experiment on {device} ---")
    print(f"Config: Dim={input_dim}, Hidden={hidden_dim}, Rank={lora_rank}")

    # 1. Setup Data & Teacher
    # Teacher is random but fixed
    teacher = MLP(input_dim, hidden_dim, input_dim).to(device)
    for p in teacher.parameters(): p.requires_grad = False
    
    # Student starts random
    student = MLP(input_dim, hidden_dim, input_dim).to(device)
    
    # Data Generator
    def get_batch(bs):
        x = torch.randn(bs, input_dim, device=device)
        with torch.no_grad():
            y = teacher(x)
        return x, y

    # 2. Phase 1: Pretraining
    print("Phase 1: Pretraining Student to match Teacher...")
    opt_pre = optim.Adam(student.parameters(), lr=lr_pre)
    loss_fn = nn.MSELoss()
    
    for ep in range(pretrain_epochs):
        ep_loss = 0
        steps = num_samples // batch_size
        for _ in range(steps):
            bx, by = get_batch(batch_size)
            pred = student(bx)
            loss = loss_fn(pred, by)
            
            opt_pre.zero_grad()
            loss.backward()
            opt_pre.step()
            ep_loss += loss.item()
        if ep % 20 == 0: print(f"  Ep {ep}: {ep_loss/steps:.5f}")

    # 3. Phase 2: The Shift
    print("Phase 2: Applying High-Rank Shift to Teacher...")
    # Create Orthogonal Matrix
    H = torch.randn(hidden_dim, hidden_dim, device=device)
    Q, _ = torch.linalg.qr(H)
    Shift = Q * 2.0 # Scale up
    
    # Apply to Teacher's middle layer
    with torch.no_grad():
        teacher.layers[1].weight.add_(Shift)
        
    # Calculate Target SR
    true_sr = calc_stable_rank(Shift)
    print(f"  Shift Applied. Target Stable Rank: {true_sr:.2f}")

    # 4. Phase 3: LoRA Finetuning
    print("Phase 3: LoRA Finetuning...")
    lora_model = LoRAMLPWrapper(student, rank=lora_rank).to(device)
    opt_ft = optim.Adam(filter(lambda p: p.requires_grad, lora_model.parameters()), lr=lr_ft)
    
    ft_loss = []
    ft_sr = []
    
    for ep in range(finetune_epochs):
        ep_loss = 0
        steps = num_samples // batch_size
        lora_model.train()
        
        for _ in range(steps):
            # Dynamic data generation (infinite stream implies generalization)
            bx, by = get_batch(batch_size) 
            
            pred = lora_model(bx)
            loss = loss_fn(pred, by)
            
            opt_ft.zero_grad()
            loss.backward()
            opt_ft.step()
            ep_loss += loss.item()
            
        # Logging
        avg_l = ep_loss/steps
        curr_sr = lora_model.get_sr()
        ft_loss.append(avg_l)
        ft_sr.append(curr_sr)
        
        if ep % 5 == 0:
            print(f"  FT Ep {ep} | Loss: {avg_l:.4f} | SR: {curr_sr:.2f}")

    # 5. Visualization
    # plt.figure(figsize=(12, 5))
    
    # plt.subplot(1, 2, 1)
    # plt.plot(ft_loss)
    # plt.title("Finetuning Loss")
    # plt.xlabel("Epoch")
    # plt.yscale("log")
    
    # plt.subplot(1, 2, 2)
    # plt.plot(ft_sr, label="LoRA SR", color='orange', linewidth=2)
    # plt.axhline(true_sr, color='red', linestyle='--', label=f"True Shift SR ({true_sr:.1f})")
    # plt.ylim(0, hidden_dim + 5)
    # plt.title("Stable Rank Analysis")
    # plt.legend()
    
    # plt.show()

In [4]:
run_experiment(
    input_dim=64,
    hidden_dim=256,
    lora_rank=4,
    num_samples=4096,
    batch_size=256,
    pretrain_epochs=300,
    finetune_epochs=100,
    lr_pre=1e-3,
    lr_ft=3e-4,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

--- Starting Toy Experiment on cuda ---
Config: Dim=64, Hidden=256, Rank=4
Phase 1: Pretraining Student to match Teacher...
  Ep 0: 0.00752
  Ep 20: 0.00176
  Ep 40: 0.00146
  Ep 60: 0.00132
  Ep 80: 0.00123
  Ep 100: 0.00118
  Ep 120: 0.00112
  Ep 140: 0.00108
  Ep 160: 0.00104
  Ep 180: 0.00099
  Ep 200: 0.00095
  Ep 220: 0.00093
  Ep 240: 0.00089
  Ep 260: 0.00087
  Ep 280: 0.00085
Phase 2: Applying High-Rank Shift to Teacher...
  Shift Applied. Target Stable Rank: 255.99
Phase 3: LoRA Finetuning...
  FT Ep 0 | Loss: 0.0897 | SR: 1.03
  FT Ep 5 | Loss: 0.0654 | SR: 1.00
  FT Ep 10 | Loss: 0.0580 | SR: 1.00
  FT Ep 15 | Loss: 0.0569 | SR: 1.00
  FT Ep 20 | Loss: 0.0567 | SR: 1.00
  FT Ep 25 | Loss: 0.0556 | SR: 1.01
  FT Ep 30 | Loss: 0.0553 | SR: 1.02
  FT Ep 35 | Loss: 0.0554 | SR: 1.03
  FT Ep 40 | Loss: 0.0544 | SR: 1.05
  FT Ep 45 | Loss: 0.0538 | SR: 1.10
  FT Ep 50 | Loss: 0.0528 | SR: 1.16
  FT Ep 55 | Loss: 0.0525 | SR: 1.24
  FT Ep 60 | Loss: 0.0518 | SR: 1.32
  FT Ep 65 | 

In [5]:
run_experiment(
    input_dim=64,
    hidden_dim=256,
    lora_rank=16,
    num_samples=4096,
    batch_size=256,
    pretrain_epochs=300,
    finetune_epochs=100,
    lr_pre=1e-3,
    lr_ft=3e-4,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

--- Starting Toy Experiment on cuda ---
Config: Dim=64, Hidden=256, Rank=16
Phase 1: Pretraining Student to match Teacher...
  Ep 0: 0.00798
  Ep 20: 0.00186
  Ep 40: 0.00156
  Ep 60: 0.00140
  Ep 80: 0.00128
  Ep 100: 0.00121
  Ep 120: 0.00115
  Ep 140: 0.00108
  Ep 160: 0.00102
  Ep 180: 0.00097
  Ep 200: 0.00092
  Ep 220: 0.00089
  Ep 240: 0.00084
  Ep 260: 0.00083
  Ep 280: 0.00082
Phase 2: Applying High-Rank Shift to Teacher...
  Shift Applied. Target Stable Rank: 255.98
Phase 3: LoRA Finetuning...
  FT Ep 0 | Loss: 0.0932 | SR: 1.10
  FT Ep 5 | Loss: 0.0583 | SR: 1.01
  FT Ep 10 | Loss: 0.0552 | SR: 1.02
  FT Ep 15 | Loss: 0.0542 | SR: 1.04
  FT Ep 20 | Loss: 0.0524 | SR: 1.09
  FT Ep 25 | Loss: 0.0493 | SR: 1.24
  FT Ep 30 | Loss: 0.0451 | SR: 1.55
  FT Ep 35 | Loss: 0.0425 | SR: 1.91
  FT Ep 40 | Loss: 0.0399 | SR: 2.23
  FT Ep 45 | Loss: 0.0372 | SR: 2.50
  FT Ep 50 | Loss: 0.0359 | SR: 2.68
  FT Ep 55 | Loss: 0.0345 | SR: 2.82
  FT Ep 60 | Loss: 0.0337 | SR: 2.91
  FT Ep 65 |

In [6]:
run_experiment(
    input_dim=64,
    hidden_dim=256,
    lora_rank=64,
    num_samples=4096,
    batch_size=256,
    pretrain_epochs=300,
    finetune_epochs=100,
    lr_pre=1e-3,
    lr_ft=3e-4,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

--- Starting Toy Experiment on cuda ---
Config: Dim=64, Hidden=256, Rank=64
Phase 1: Pretraining Student to match Teacher...
  Ep 0: 0.00776
  Ep 20: 0.00174
  Ep 40: 0.00148
  Ep 60: 0.00135
  Ep 80: 0.00126
  Ep 100: 0.00121
  Ep 120: 0.00115
  Ep 140: 0.00109
  Ep 160: 0.00106
  Ep 180: 0.00100
  Ep 200: 0.00096
  Ep 220: 0.00092
  Ep 240: 0.00091
  Ep 260: 0.00087
  Ep 280: 0.00085
Phase 2: Applying High-Rank Shift to Teacher...
  Shift Applied. Target Stable Rank: 255.99
Phase 3: LoRA Finetuning...
  FT Ep 0 | Loss: 0.1001 | SR: 1.06
  FT Ep 5 | Loss: 0.0585 | SR: 1.04
  FT Ep 10 | Loss: 0.0526 | SR: 1.23
  FT Ep 15 | Loss: 0.0445 | SR: 1.83
  FT Ep 20 | Loss: 0.0373 | SR: 2.60
  FT Ep 25 | Loss: 0.0325 | SR: 3.26
  FT Ep 30 | Loss: 0.0291 | SR: 3.73
  FT Ep 35 | Loss: 0.0261 | SR: 4.08
  FT Ep 40 | Loss: 0.0242 | SR: 4.25
  FT Ep 45 | Loss: 0.0226 | SR: 4.38
  FT Ep 50 | Loss: 0.0215 | SR: 4.48
  FT Ep 55 | Loss: 0.0204 | SR: 4.57
  FT Ep 60 | Loss: 0.0197 | SR: 4.63
  FT Ep 65 |

In [7]:
run_experiment(
    input_dim=64,
    hidden_dim=256,
    lora_rank=256,
    num_samples=4096,
    batch_size=256,
    pretrain_epochs=300,
    finetune_epochs=100,
    lr_pre=1e-3,
    lr_ft=3e-4,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

--- Starting Toy Experiment on cuda ---
Config: Dim=64, Hidden=256, Rank=256
Phase 1: Pretraining Student to match Teacher...
  Ep 0: 0.00870
  Ep 20: 0.00185
  Ep 40: 0.00154
  Ep 60: 0.00138
  Ep 80: 0.00129
  Ep 100: 0.00123
  Ep 120: 0.00116
  Ep 140: 0.00110
  Ep 160: 0.00106
  Ep 180: 0.00101
  Ep 200: 0.00096
  Ep 220: 0.00092
  Ep 240: 0.00088
  Ep 260: 0.00085
  Ep 280: 0.00083
Phase 2: Applying High-Rank Shift to Teacher...
  Shift Applied. Target Stable Rank: 255.98
Phase 3: LoRA Finetuning...
  FT Ep 0 | Loss: 0.0797 | SR: 1.04
  FT Ep 5 | Loss: 0.0456 | SR: 2.12
  FT Ep 10 | Loss: 0.0308 | SR: 4.30
  FT Ep 15 | Loss: 0.0251 | SR: 5.20
  FT Ep 20 | Loss: 0.0219 | SR: 5.64
  FT Ep 25 | Loss: 0.0200 | SR: 6.04
  FT Ep 30 | Loss: 0.0186 | SR: 6.44
  FT Ep 35 | Loss: 0.0174 | SR: 6.62
  FT Ep 40 | Loss: 0.0166 | SR: 6.41
  FT Ep 45 | Loss: 0.0158 | SR: 6.33
  FT Ep 50 | Loss: 0.0155 | SR: 6.33
  FT Ep 55 | Loss: 0.0152 | SR: 6.40
  FT Ep 60 | Loss: 0.0148 | SR: 6.42
  FT Ep 65 